# ETL fundos

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import requests
from datetime import datetime
from dateutil.relativedelta import relativedelta
import io
import zipfile

In [4]:
ano_mes = "202508"
caminho_fundos = Path("fundos")

caminho_fundos.mkdir(exist_ok=True)

# while ano_mes != "202608":
#     response = requests.get(rf"https://dados.cvm.gov.br/dados/FI/DOC/CDA/DADOS/cda_fi_{ano_mes}.zip", stream=True)

#     if response.status_code == 200:
#         with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
#             arquivos = zip_ref.namelist()
            
#             bloco_2_lista= [arq for arq in arquivos if "BLC_2" in arq]

#             if bloco_2_lista:
#                 bloco_2 = bloco_2_lista[0]
#                 zip_ref.extract(bloco_2, path=caminho_fundos)
#                 print(f"{bloco_2} baixado com sucesso\n")
#             else:
#                 print(f"nenhum arquivo relacionado foi encontrado em {ano_mes}")
#     else:
#         print(f"ERRO: falha no mÊs {ano_mes}: {response.status_code}")
#     ano_mes = datetime.strptime(ano_mes, "%Y%m")

#     ano_mes += relativedelta(months=1)

#     ano_mes = ano_mes.strftime('%Y%m')    
            

In [5]:
df_fundos_def = pd.DataFrame()
for fundo in caminho_fundos.glob('*.csv'):
    df_fundo = pd.read_csv(fundo, encoding='latin1', sep=';', dtype={'TP_NEGOC': str})
    cols_importantes = ["DT_COMPTC", "CNPJ_FUNDO_CLASSE", "DENOM_SOCIAL","TP_APLIC", "CNPJ_FUNDO_CLASSE_COTA", "TP_ATIVO","NM_FUNDO_CLASSE_SUBCLASSE_COTA", "VL_MERC_POS_FINAL"]
    df_fundo = df_fundo[cols_importantes]
    df_fundo = df_fundo[df_fundo['VL_MERC_POS_FINAL'] >= 100000]
    df_fundos_def = pd.concat([df_fundos_def, df_fundo], axis=0, ignore_index=True)

df_fundos_def.shape

(1203043, 8)

In [6]:
df_fundos_def['DT_COMPTC'].value_counts()

DT_COMPTC
2025-12-31    114015
2026-01-31    113864
2025-08-31    113772
2025-09-30    113732
2025-11-30    113015
2025-10-31    112823
2026-03-31    112360
2026-02-28    112030
2026-04-30    109511
2026-05-31     71343
2026-06-30     69870
2026-07-31     46708
Name: count, dtype: int64

# DF -> Grafo

In [7]:
import networkx as nx

In [8]:
df_fundos_def['DT_COMPTC'] = pd.to_datetime(df_fundos_def['DT_COMPTC'])

In [10]:
resultados_centralidade = []

"""
esse for irá "fatiar" o df pelos meses
a var mes vai guardar o "rótulo" do mês (ex: 05-2026)
dados_mes é o df em si
"""
for mes, dados_mes in df_fundos_def.groupby(df_fundos_def['DT_COMPTC'].dt.to_period('M')):

    # construção do grafo direcional
    G_mes = nx.from_pandas_edgelist(
        dados_mes,
        source='CNPJ_FUNDO_CLASSE',
        target="CNPJ_FUNDO_CLASSE_COTA",
        edge_attr="VL_MERC_POS_FINAL",
        create_using=nx.DiGraph()
    )

    # cálculo da centralidade de grau (métrica Efeito Manada)
    in_degree = nx.in_degree_centrality(G_mes)

    # Métrica de Risco de Contágio (Quem distribui investimento)
    out_degree = nx.out_degree_centrality(G_mes)

    # armazenando os resultados
    for cnpj in G_mes.nodes():
        resultados_centralidade.append({
            "Mes" : str(mes),
            "CNPJ" : cnpj,
            "Central_Grau_Ent" : in_degree[cnpj],
            "Central_Grau_Saida" : out_degree[cnpj]
        })

In [11]:
# matriz de features
df_features = pd.DataFrame(resultados_centralidade)
df_features = df_features.sort_values(by=['CNPJ', 'Mes']).reset_index(drop=True)

In [13]:
df_features.head(10)

,Mes,CNPJ,Central_Grau_Ent,Central_Grau_Saida
0,2025-08,00.068.305/0001-35,0.0,0.000038
1,2025-09,00.068.305/0001-35,0.0,0.000037
2,2025-10,00.068.305/0001-35,0.0,0.000038
3,2025-11,00.068.305/0001-35,0.0,0.000037
4,2025-12,00.068.305/0001-35,0.0,0.000037
5,2026-01,00.068.305/0001-35,0.0,0.000037
6,2026-02,00.068.305/0001-35,0.0,0.000038
7,2026-03,00.068.305/0001-35,0.0,0.000038
8,2026-04,00.068.305/0001-35,0.0,0.000038
9,2026-05,00.068.305/0001-35,0.0,0.000048


# Modelagem ML

## Isolation Forest

In [21]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

In [18]:
# transformando meses em colunas
df_ml = df_features.pivot(index='CNPJ',
                          columns='Mes',
                          values=['Central_Grau_Ent', 'Central_Grau_Saida']).fillna(0)

df_ml.columns = [f"{metrica}_{mes}" for metrica, mes in df_ml.columns]

df_ml.shape

(30449, 24)

In [19]:
df_ml = df_ml.loc[~(df_ml.select_dtypes(include='number') == 0).all(axis=1)]
df_ml.shape

(30449, 24)

In [20]:
colunas_manada = [col for col in df_ml.columns if 'Ent' in col]
colunas_contagio = [col for col in df_ml.columns if 'Saida' in col]

df_manada = df_ml[colunas_manada]
df_contagio = df_ml[colunas_contagio]

## Efeito Manada

In [ ]:
# config do modelo IF
modelo_if = IsolationForest(n_estimators=100, contamination=0.01, random_state=42)
mo

df_ml['Previsao_Anomalia_if'] = modelo_if.fit_predict(df_ml)

In [71]:
# marcando apenas os fundos que o modelo marcou como risco
df_anomalias = df_ml[df_ml['Previsao_Anomalia_if'] == -1]
df_anomalias

Mes,2026-05,2026-06,2026-07,Previsao_Anomalia_if
CNPJ,,,,
03.256.793/0001-00,0.005477,0.005042,0.006540,-1
03.504.023/0001-21,0.006592,0.006772,0.009469,-1
04.699.738/0001-40,0.005768,0.005882,0.006267,-1
05.584.551/0001-63,0.005962,0.005981,0.005586,-1
05.786.931/0001-80,0.004314,0.004449,0.005927,-1
...,...,...,...,...
62.400.417/0001-07,0.005380,0.005783,0.005722,-1
62.912.973/0001-54,0.003296,0.004152,0.004837,-1
63.601.544/0001-29,0.007416,0.009144,0.009878,-1


In [72]:
print(f"O modelo analisou {len(df_ml)} fundos únicos")
print(f"Foram detectadas {len(df_anomalias)} anomalias de comportamento")
print("\n--- Fundos Anômalos ---")
print(df_anomalias.head(10))

O modelo analisou 11079 fundos únicos
Foram detectadas 111 anomalias de comportamento

--- Fundos Anômalos ---
Mes                  2026-05   2026-06   2026-07  Previsao_Anomalia_if
CNPJ                                                                  
03.256.793/0001-00  0.005477  0.005042  0.006540                    -1
03.504.023/0001-21  0.006592  0.006772  0.009469                    -1
04.699.738/0001-40  0.005768  0.005882  0.006267                    -1
05.584.551/0001-63  0.005962  0.005981  0.005586                    -1
05.786.931/0001-80  0.004314  0.004449  0.005927                    -1
06.175.696/0001-73  0.013766  0.014334  0.014374                    -1
07.096.546/0001-37  0.012893  0.012852  0.013012                    -1
07.103.364/0001-46  0.004362  0.004449  0.003883                    -1
07.187.542/0001-64  0.003635  0.003707  0.005041                    -1
07.408.525/0001-00  0.007271  0.007365  0.010082                    -1


## LOF (Local Outlier Factor)

In [73]:
from sklearn.neighbors import LocalOutlierFactor

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.01)
df_ml['Previsao_anomalia_lof'] = lof.fit_predict(df_ml)

In [74]:
df_ml.value_counts('Previsao_anomalia_lof')

Previsao_anomalia_lof
 1    10968
-1      111
Name: count, dtype: int64

In [82]:
# Criando uma máscara de consenso: True apenas se ambos os modelos deram -1
df_ml['Alerta_Maximo'] = (df_ml['Previsao_Anomalia_if'] == -1) & (df_ml['Previsao_anomalia_lof'] == -1)

# Filtrando apenas os fundos de altíssimo risco
df_consenso = df_ml[df_ml['Alerta_Maximo']]

print(f"Quantidade de fundos apontados por AMBOS os modelos: {len(df_consenso)}")

Quantidade de fundos apontados por AMBOS os modelos: 0


In [85]:
df_anomalias_lof = df_ml[df_ml['Previsao_anomalia_lof'] == -1]
df_anomalias_lof.head(10)

Mes,2026-05,2026-06,2026-07,Previsao_Anomalia_if,Previsao_anomalia_lof,Alerta_Maximo
CNPJ,,,,,,
01.591.605/0001-67,0.000291,0.000247,0.000000,1,-1,False
03.394.711/0001-86,0.000339,0.000346,0.000545,1,-1,False
06.128.183/0001-01,0.000291,0.000297,0.000477,1,-1,False
06.916.384/0001-73,0.000194,0.000247,0.000341,1,-1,False
07.725.538/0001-02,0.000097,0.000000,0.000136,1,-1,False
08.680.812/0001-37,0.000242,0.000297,0.000341,1,-1,False
08.893.167/0001-30,0.000097,0.000000,0.000136,1,-1,False
10.309.539/0001-80,0.000242,0.000346,0.000341,1,-1,False
14.180.011/0001-05,0.000339,0.000297,0.000477,1,-1,False
